In [1]:
import pandas as pd
import re
from collections import defaultdict
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

import os
def get_indiv_concepts(formula) -> list:
    concepts = []
    concps = re.findall(r'(?<!\bNOT\s)(?:\b(?:hyp|pre|oth):[^\s)]+)', formula)
    for c in concps:
        try:
            end_idx = c.index(')')
        except:
            end_idx = len(c)
        concepts.append(c[:end_idx])
    return concepts



def load_csv_data(filepath):
    """Load CSV and extract unit-concept mappings."""
    df = pd.read_csv(filepath)
    unit_concepts = defaultdict(set)
    raw_concepts= []
    for _, row in df.iterrows():
        unit = row['unit']
        formula = row['best_name']
        concepts = get_indiv_concepts(formula)
        
        unit_concepts[unit].update(concepts)
        raw_concepts.extend(concepts)
        
    return unit_concepts, raw_concepts
#how concepts removal same across algortihm


In [2]:
def get_all_cps_for_pi(folder):
    root_path = Path(folder)

    # Find all matching CSV files
    csv_pattern = 'Cluster*IOUS1024N.csv'
    csv_files = list(root_path.rglob(csv_pattern))
    
    concept_dict=defaultdict(set)
    for csv_file in csv_files:
        concepts = []
        csv_file = os.path.join(folder, csv_file)
        df = pd.read_csv(csv_file)
        for unit, formula in zip(df.unit, df.best_name):
            concept_dict[unit].update(get_indiv_concepts(formula))
    l=0
    un=set()
    for u, c in concept_dict.items():
        un.update(c)
    print(folder, len(un))
    return concept_dict

In [108]:
dummy={9: {'a':100, 'b':9}, 10: {'v':9, 'j':89},8:{'l':89, 'k':89} }
d=dict(
    sorted(
        dummy.items(),
        key=lambda x: sum(x[1].values()),
        reverse=True
    )
)
d

{8: {'l': 89, 'k': 89}, 9: {'a': 100, 'b': 9}, 10: {'v': 9, 'j': 89}}

## **Concept redundancy**

In [104]:
import pickle
with open("/workspace/CCE_NLI/code/Abstractions/final_abstractions.pkl", 'rb') as f:
    abs_map = pickle.load(f)

def find_cluster(raw_concept):
    for cluster in abs_map:
        if raw_concept in abs_map[cluster]:
            return cluster
def find_abstractions(expls):
    if isinstance(expls, set):
        glob = list(expls)
    else:
        glob=expls
    abstracts=[]
    exact_concepts_perabs=defaultdict(list)
    for concept in glob:
        raw_concept = concept.split(":")[-1]
        abstraction_cluster = find_cluster(raw_concept)
        if abstraction_cluster==143: 
            continue
        if not abstraction_cluster:abstraction_cluster=150
        exact_concepts_perabs[abstraction_cluster].append(raw_concept)
        
        abstracts.append(abstraction_cluster)
    for a in exact_concepts_perabs:
        exact_concepts_perabs[a] = Counter(exact_concepts_perabs[a])
    return Counter(abstracts), set(abstracts), dict(
                                                    sorted(
                                                        exact_concepts_perabs.items(),
                                                        key=lambda x: sum(x[1].values()),
                                                        reverse=True
                                                    )
)


In [109]:
import os
from collections import Counter, defaultdict
import matplotlib.pyplot as plt

def load_csv_data_safe(filepath):
    """Load CSV if it exists, else return empty lists."""
    if os.path.exists(filepath):
        return load_csv_data(filepath)  # your existing loader
    else:
        print(f"Warning: {filepath} not found. Skipping.")
        return [], []

methods = ['lottery_ticket', 'wanda']
global_red = defaultdict(list)

# Two roots per method
roots = ['Run0.25_5/Expls'] #'Run0.25_6/Expls', 'Run0.25_7/Expls']
print("In % of the clus")
for method in methods:
    global_avg_redundancy_at_sparsity = []
    local_avg_redundancy_at_sparsity = []

    # Get all sparsity folders (union across both roots)
    sparsity_folders = set()
    for root_sub in roots:
        full_root = f'/workspace/CCE_NLI/LLAMA/exp/{method}/{root_sub}'
        if os.path.exists(full_root):
            sparsity_folders.update(os.listdir(full_root))

    sparsity_folders = sorted(sparsity_folders)
    print(sparsity_folders)
    for sparsity in sparsity_folders:
        print(sparsity)
        if '0.0%Pruned' != sparsity and '68' not in sparsity: continue
        if '.ipy' in sparsity: continue
        avg_redundancy_at_cluster = []
        global_concepts = []
        cluster_concepts = []
        for cluster in range(1, 4):
            cluster_concepts = []
            

            # Loop over the two roots
            for root_sub in roots:
                if sparsity=='0.0%Pruned':
                    root_sub='Run0.25_5/Expls'
                filepath = f'/workspace/CCE_NLI/LLAMA/exp/{method}/{root_sub}/{sparsity}/Cluster{cluster}IOUS1024N.csv'
                _, concepts = load_csv_data_safe(filepath)
                if concepts:
                    cluster_concepts.extend(concepts)
            global_concepts.extend(cluster_concepts)
            sorted_dict_asc = {k: v for k, v in sorted(Counter(cluster_concepts).items(), key=lambda item: item[1],reverse=True)}
            
            abstractfreq, absnum, exactcp =find_abstractions(cluster_concepts)
            abstractfreq = {k: v for k, v in sorted(abstractfreq.items(), key=lambda item: item[1],reverse=True)}
            
            print(cluster, method )
            print({c: np.round(100*f/sum(list(sorted_dict_asc.values())),3) for c,f in sorted_dict_asc.items() if f>1 and ":tag:" not in c})
            ct=0
            avg=0
            for j in abstractfreq:
                if len(exactcp[j]) >1:
                    print(f"{j} : {c:f/sum(list(sorted_dict_asc.values())) for c,f in exactcp[j].items()}")
                    avg += len(exactcp[j])
                    ct +=1
            print(avg, ct)
           
           
        
                    
            print()
            print()
            print()
            print()

            if cluster_concepts:
                counter = Counter(cluster_concepts)
        
                
                avg_redundancy_at_cluster.append(sum(counter.values()) / len(counter))
       

        # Global redundancy
        if global_concepts:
            counter = Counter(global_concepts)
            global_avg_redundancy_at_sparsity.append(sum(counter.values()) / len(counter))
            

        local_avg_redundancy_at_sparsity.append(avg_redundancy_at_cluster)

    # Save results
    global_red[method] = global_avg_redundancy_at_sparsity

    # --- Plot Local Redundancy ---
   
   
# --- Plot Global Redundancy ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
print("Global avg:", global_red)

x1 = range(len(global_red[methods[0]]))
ax1.plot(x1, global_red[methods[0]], marker='o', label=methods[0])
ax1.set_title(f"Global Redundancy {methods[0]}")
ax1.set_xlabel("Index")
ax1.set_ylabel("Global Avg")
ax1.legend()
ax1.grid()

x2 = range(len(global_red[methods[1]]))
ax2.plot(x2, global_red[methods[1]], marker='s', label=methods[1], color='orange')
ax2.set_title(f"Global Redundancy {methods[1]}")
ax2.set_xlabel("Index")
ax2.set_ylabel("Global Avg")
ax2.legend()
ax2.grid()
plt.show()


In % of the clus
['0.0%Pruned', '25.0%Pruned', '43.75%Pruned', '57.813%Pruned', '68.359%Pruned', '76.27%Pruned']
0.0%Pruned
1 lottery_ticket
{'oth:overlap:overlap25': 6.044, 'pre:tok:man': 2.348, 'pre:tok:sitting': 1.349, 'hyp:tok:sitting': 1.149, 'hyp:tok:man': 1.149, 'hyp:tok:to': 1.099, 'pre:tok:woman': 1.049, 'hyp:tok:men': 0.949, 'pre:tok:men': 0.899, 'oth:overlap:overlap50': 0.899, 'hyp:tok:people': 0.749, 'hyp:tok:woman': 0.749, 'pre:tok:to': 0.649, 'pre:tok:walking': 0.649, 'hyp:tok:walking': 0.649, 'hyp:tok:outside': 0.599, 'hyp:tok:are': 0.599, 'pre:tok:wearing': 0.549, 'pre:tok:boy': 0.549, 'pre:tok:dog': 0.5, 'pre:tok:people': 0.45, 'oth:overlap:overlap75': 0.4, 'pre:tok:his': 0.4, 'hyp:tok:for': 0.35, 'pre:tok:in': 0.35, 'pre:tok:person': 0.3, 'pre:tok:and': 0.3, 'hyp:tok:next': 0.3, 'hyp:tok:wearing': 0.3, 'pre:tok:at': 0.25, 'hyp:tok:running': 0.25, 'pre:tok:street': 0.25, 'pre:tok:running': 0.2, 'hyp:tok:young': 0.2, 'pre:tok:sits': 0.2, 'hyp:tok:dog': 0.2, 'pre:tok:on'

TypeError: unsupported operand type(s) for /: 'Counter' and 'int'

In [37]:
import matplotlib.pyplot as plt

# transpose to get 3 lines

#do concept red but instead of concepts alone get groups of concetps are save (like get infiv concepts to get groups) and couint red of groups

## Tracking explainable nueron count

In [ ]:

def load_neuron_data(filepath):
    """Load CSV and extract unit-concept mappings."""
    df = pd.read_csv(filepath)
    explainable_units = []
    raw_concepts= []
    for _, row in df.iterrows():
        unit = row['unit']
        explainable_units.append(unit)
        
    
    return explainable_units
base = '/workspace/CCE_NLI/BERT/exp/CoFi/Run0.25_new/Expls/'
tracking_expl_units= defaultdict(lambda: defaultdict())
for sparsity in os.listdir(base):
    try:
    
        sparsity = float(sparsity.split("%")[0])
    except:
        continue
    for cluster in ['Cluster1IOUS1024N.csv', 'Cluster2IOUS1024N.csv', 'Cluster3IOUS1024N.csv']:
        filepath = os.path.join(base, f'{sparsity}%Pruned', cluster)
      
        try:
            explble_units = load_neuron_data(filepath)
            tracking_expl_units[str(sparsity)][cluster] = len(explble_units)
        except:
            continue
        
        
tracking_expl_units

## Net gain in func non func concepts per sparsity cluster wise and how it affects number and % of func and non func

In [ ]:
#clusterwise but not unique to each cluster
#cluster wise
import os

base_dir = "/workspace/CCE_NLI/BERT/exp/CoFi/Run0.25_new/Expls/"

cluster_files = [
    "Cluster1IOUS1024N.csv",
    "Cluster2IOUS1024N.csv",
    "Cluster3IOUS1024N.csv",
]

critical_tokens = [
    "a", "an", "the", "this", "that", "these", "those",
    "some", "any", "each", "every", "no",
    "i", "you", "he", "she", "it", "they",
    "me", "him", "her", "them",
    "his", "hers", "their", "its",
    "someone", "something", "nobody",
    "in", "on", "at", "by", "for", "from", "to", "with",
    "about", "into", "through", "over", "under",
    "between", "near", "inside", "outside",
    "and", "or", "but", "because", "while", "if", "though", "although", "so",
    "is", "am", "are", "was", "were", "be", "being", "been",
    "do", "does", "did",
    "have", "has", "had",
    "can", "could", "will", "would", "should", "may", "might", "must",
    "not", "none", "never", "nothing", "least", "most", "all",
    "there", "here", "just", "only", "also", "very",
    ".", ",", ":", ";", "!", "?"
]

def label_concept(concept: str):
    """Return 'c' for critical or 'nc' for non-critical"""
    if ":tag:" in concept:
        return "c"
    if concept.startswith("oth:"):
        return "c"
    if ":tok:" in concept:
        token = concept.split(":tok:")[-1]
        return "c" if token in critical_tokens else "nc"
    return "nc"

def load_cluster_concepts(sparsity_dir, cluster_file):
    """Load concepts from a single cluster CSV"""
    try:
        
        unit_concepts = load_csv_data(os.path.join(sparsity_dir, cluster_file))
       
        all_concepts = set()
        for concepts in unit_concepts.values():
            all_concepts.update(concepts)
        return all_concepts
    except Exception:
        return set()

# Load concepts per cluster per sparsity
sparsity_to_cluster_concepts = {}  # sparsity -> cluster -> set(concepts)

for sparsity in os.listdir(base_dir):
    sparsity_path = os.path.join(base_dir, sparsity)
    
    try:
        s = float(sparsity.split("%")[0])
   
    except ValueError:
        continue

    cluster_concepts = {}
    for i, fname in enumerate(cluster_files, 1):
        cluster_name = f"c{i}"
        concepts = load_cluster_concepts(sparsity_path, fname)
        cluster_concepts[cluster_name] = concepts
    

    sparsity_to_cluster_concepts[s] = cluster_concepts
    total = sum(len(c) for c in cluster_concepts.values())
    print(f"Sparsity {s}: {total} total concepts across clusters")

# Track added/removed functional concepts per cluster between sparsity steps
removed_concepts_by_step = {}
added_concepts_by_step = {}

sparsities = sorted(sparsity_to_cluster_concepts.keys())
for i in range(-1, len(sparsities) - 1):
    s_prev, s_next = sparsities[0], sparsities[i + 1]

    print(f"\nSparsity {s_prev} → {s_next}")

    for cluster in cluster_files:
        c_name = f"c{cluster_files.index(cluster) + 1}"
        prev = sparsity_to_cluster_concepts[s_prev][c_name]
        next_ = sparsity_to_cluster_concepts[s_next][c_name]

        removed = prev - next_
        added = next_ - prev

        removed_func = sum(1 for c in removed if label_concept(c) == 'c')
        added_func = sum(1 for c in added if label_concept(c) == 'c')

        print(f"\nCluster {c_name}:")
        #print(f"  Removed concepts: {len(removed)} (Functional: {removed_func}): {removed}")
        #print(f"  Added concepts:   {len(added)} (Functional: {added_func}): {added}")
        #print(f"  Prev Total concepts:   {len(prev)}")
        print(f"  Next concepts:   {next_}")
        
        new_func = sum(1 for c in next_ if label_concept(c) == 'c')
        if len(next_) >0:
            #print(f"  % of func:   {new_func}/{len(next_)} = {new_func/len(next_)}")
            #print(f"  % of non func:   {len(next_) - new_func}/{len(next_)} = {1 - (new_func/len(next_))}")
            pass

        removed_concepts_by_step[(s_prev, s_next, c_name)] = removed
        added_concepts_by_step[(s_prev, s_next, c_name)] = added


## net gain in number and % of func and non func unique to a cluster and how that changes func and nonfunc distribution

In [ ]:
#cluster wise
import os

base_dir = "/workspace/CCE_NLI/BERT/exp/CoFi/Run0.25_new/Expls/"

cluster_files = [
    "Cluster1IOUS1024N.csv",
    "Cluster2IOUS1024N.csv",
    "Cluster3IOUS1024N.csv",
]

critical_tokens = [
    "a", "an", "the", "this", "that", "these", "those",
    "some", "any", "each", "every", "no",
    "i", "you", "he", "she", "it", "they",
    "me", "him", "her", "them",
    "his", "hers", "their", "its",
    "someone", "something", "nobody",
    "in", "on", "at", "by", "for", "from", "to", "with",
    "about", "into", "through", "over", "under",
    "between", "near", "inside", "outside",
    "and", "or", "but", "because", "while", "if", "though", "although", "so",
    "is", "am", "are", "was", "were", "be", "being", "been",
    "do", "does", "did",
    "have", "has", "had",
    "can", "could", "will", "would", "should", "may", "might", "must",
    "not", "none", "never", "nothing", "least", "most", "all",
    "there", "here", "just", "only", "also", "very",
    ".", ",", ":", ";", "!", "?"
]

def label_concept(concept: str):
    """Return 'c' for critical or 'nc' for non-critical"""
    if ":tag:" in concept:
        return "c"
    if concept.startswith("oth:"):
        return "c"
    if ":tok:" in concept:
        token = concept.split(":tok:")[-1]
        return "c" if token in critical_tokens else "nc"
    return "nc"

def load_cluster_concepts(sparsity_dir, cluster_file):
    """Load concepts from a single cluster CSV"""
    try:
        unit_concepts = load_csv_data(os.path.join(sparsity_dir, cluster_file))
        all_concepts = set()
        for concepts in unit_concepts.values():
            all_concepts.update(concepts)
        return all_concepts
    except Exception:
        return set()

# Load concepts per cluster per sparsity
sparsity_to_cluster_concepts_unique = {}  # sparsity -> cluster -> set(concepts)

for sparsity in os.listdir(base_dir):
    sparsity_path = os.path.join(base_dir, sparsity)
    try:
        s = float(sparsity.split("%")[0])
    except ValueError:
        continue

    cluster_concepts = {}
    for i, fname in enumerate(cluster_files, 1):
        cluster_name = f"c{i}"
        concepts = load_cluster_concepts(sparsity_path, fname)
        cluster_concepts[cluster_name] = concepts
    
    c1o = cluster_concepts['c1']  - cluster_concepts['c2'] - cluster_concepts['c3']  
    c2o = cluster_concepts['c2']  - cluster_concepts['c1'] - cluster_concepts['c3']  
    c3o = cluster_concepts['c3']  - cluster_concepts['c1'] - cluster_concepts['c2']  
    cluster_concepts['c1']=c1o
    cluster_concepts['c2']=c2o
    cluster_concepts['c3']=c3o
    sparsity_to_cluster_concepts_unique[s] = cluster_concepts
    total = sum(len(c) for c in cluster_concepts.values())
    print(f"Sparsity {s}: {total} total concepts across clusters")

# Track added/removed functional concepts per cluster between sparsity steps
removed_concepts_by_step = {}
added_concepts_by_step = {}

sparsities = sorted(sparsity_to_cluster_concepts_unique.keys())
for i in range(-1, len(sparsities) - 1):
    s_prev, s_next = sparsities[0], sparsities[i + 1]

    print(f"\nSparsity {s_prev} → {s_next}")

    for cluster in cluster_files:
        c_name = f"c{cluster_files.index(cluster) + 1}"
        prev = sparsity_to_cluster_concepts_unique[s_prev][c_name]
        next_ = sparsity_to_cluster_concepts_unique[s_next][c_name]

        removed = prev - next_
        added = next_ - prev

        removed_func = sum(1 for c in removed if label_concept(c) == 'c')
        added_func = sum(1 for c in added if label_concept(c) == 'c')

        print(f"\nCluster {c_name}:")
        print(f"  Removed concepts: {len(removed)} (Functional: {removed_func})")
        print(f"  Added concepts:   {len(added)} (Functional: {added_func})")
        print(f"  Prev Total concepts:   {len(prev)}")
        print(f"  Next Total concepts:   {len(next_)}")
        
        new_func = sum(1 for c in next_ if label_concept(c) == 'c')
        print(f"  % of func:   {new_func}/{len(next_)} = {new_func/len(next_)}")
        print(f"  % of non func:   {len(next_) - new_func}/{len(next_)} = {1 - (new_func/len(next_))}")

        removed_concepts_by_step[(s_prev, s_next, c_name)] = removed
        added_concepts_by_step[(s_prev, s_next, c_name)] = added


## Plotting % and number of concepts (unique to cluster and cluster-wise)

In [ ]:
import matplotlib.pyplot as plt

def plot_func_percentages(sparsity_to_cluster_concepts, unique=False):
    """
    Plot percent of functional concepts per cluster vs sparsity.

    Parameters:
    - sparsity_to_cluster_concepts: dict[sparsity -> cluster -> set(concepts)]
    - unique: if True, counts only concepts unique to that cluster
    """
    sparsities = sorted(sparsity_to_cluster_concepts.keys())
    clusters = ['c1', 'c2', 'c3']

    # Prepare data for plotting
    func_percent = {c: [] for c in clusters}

    for s in sparsities:
        if s > 0.6: continue
        cluster_concepts = sparsity_to_cluster_concepts[s]

        # If unique=True, remove concepts that appear in other clusters at same sparsity
        if unique:
            all_concepts = set().union(*cluster_concepts.values())
            unique_concepts = {}
            for c in clusters:
                others = all_concepts - cluster_concepts[c]
                unique_concepts[c] = cluster_concepts[c] - others
            cluster_concepts = unique_concepts

        for c in clusters:
            concepts = cluster_concepts[c]
            if len(concepts) == 0:
                pct = 0
            else:
                func_count = sum(1 for x in concepts if label_concept(x) == 'c')
                pct = func_count / len(concepts)
            func_percent[c].append(pct * 100)  # convert to %
    
    # Plot
    plt.figure(figsize=(8,6))
    for c in clusters:
        plt.plot(sparsities[:-2], func_percent[c], marker='o', label=f"{c}")
    plt.xlabel("Sparsity")
    plt.ylabel("% Functional Concepts")
    title = "Percent of Functional Concepts per Cluster"
    if unique:
        title += " (Unique to Cluster)"
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()
plot_func_percentages(sparsity_to_cluster_concepts_unique, unique=True)
plot_func_percentages(sparsity_to_cluster_concepts, unique=False)


these graphs say that: overall, pruning induces more generality in the final layer, but nonfunc concepts still become more prevalent at c3

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_func_vs_total_grid(sparsity_to_cluster_concepts, unique=False):
    """
    Plot 4 subplots in a 1-row, 4-column grid showing functional vs total concepts per cluster,
    with numbers on top.
    
    Parameters:
    - sparsity_to_cluster_concepts: dict[sparsity -> cluster -> set(concepts)]
    - unique: if True, counts only concepts unique to that cluster
    """
    sparsities = sorted(sparsity_to_cluster_concepts.keys())[:4]  # pick first 4 sparsities
    clusters = ['c1', 'c2', 'c3']
    n_clusters = len(clusters)
    width = 0.35
    
    # 1 row, 4 columns
    fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharey=True)
    
    for i, s in enumerate(sparsities):
        ax = axes[i]
        
        cluster_concepts = sparsity_to_cluster_concepts[s]
        
        # Unique-to-cluster filter
        if unique:
            all_concepts = set().union(*cluster_concepts.values())
            unique_concepts = {}
            for c in clusters:
                others = all_concepts - cluster_concepts[c]
                unique_concepts[c] = cluster_concepts[c] - others
            cluster_concepts = unique_concepts
        
        func_counts = []
        total_counts = []
        for c in clusters:
            concepts = cluster_concepts[c]
            total_counts.append(len(concepts))
            func_counts.append(sum(1 for x in concepts if label_concept(x) == 'c'))
        
        x = np.arange(n_clusters)
        bars_func = ax.bar(x - width/2, func_counts, width, label='Functional', color='skyblue')
        bars_total = ax.bar(x + width/2, total_counts, width, label='Total', color='orange')
        
        # Numbers on top
        for bar in bars_func + bars_total:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, height + 0.5, f'{int(height)}',
                    ha='center', va='bottom', fontsize=10)
        
        ax.set_xticks(x)
        ax.set_xticklabels(clusters)
        ax.set_xlabel("Cluster")
        ax.set_title(f"Sparsity {s}")
        ax.grid(axis='y', linestyle='--', alpha=0.5)
    
    axes[0].set_ylabel("Number of Concepts")
    fig.suptitle("Functional vs Total Concepts per Cluster", fontsize=16)
    fig.legend(['Functional', 'Total'], loc='upper right')
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

# Call the function
plot_func_vs_total_grid(sparsity_to_cluster_concepts, unique=False)
plot_func_vs_total_grid(sparsity_to_cluster_concepts_unique, unique=True)

## Concept redundancy func vs non func number wise (AVG concept redundancy)

In [ ]:
import os
from collections import defaultdict, Counter

def load_cluster_concepts_from_csv(sparsity_dir):
    """
    Load concepts for c1, c2, c3 from CSVs in a given sparsity folder.
    Returns dict: cluster_name -> list(concepts)
    """
    cluster_files = [
        "Cluster1IOUS1024N.csv",
        "Cluster2IOUS1024N.csv",
        "Cluster3IOUS1024N.csv",
    ]
    cluster_concepts = {}
    for i, fname in enumerate(cluster_files, start=1):
        fpath = os.path.join(sparsity_dir, fname)
        try:
            unit_concepts = load_csv_data(fpath)  # your function: unit -> set(concepts)
            all_concepts = []
            for concepts in unit_concepts.values():
                all_concepts.extend(concepts)
            cluster_concepts[f'c{i}'] = all_concepts
        except Exception as e:
            print(f"Failed to load {fpath}: {e}")
            cluster_concepts[f'c{i}'] = []
    return cluster_concepts

def compute_func_nonfunc_redundancy(base_dir):
    """
    Load CSVs for each sparsity and cluster, compute average redundancy for functional vs non-functional concepts.
    
    Returns:
    - redundancy_stats[sparsity][cluster] = {
        'func_total': number of unique functional concepts,
        'func_avg_redundancy': average times each functional concept appears across clusters,
        'nonfunc_total': number of unique non-functional concepts,
        'nonfunc_avg_redundancy': average times each non-functional concept appears across clusters
      }
    """
    redundancy_stats = {}

    for sparsity in os.listdir(base_dir):
        sparsity_path = os.path.join(base_dir, sparsity)
        try:
            s = float(sparsity.split("%")[0])
            if s >0.6: continue
        except ValueError:
            continue

        cluster_concepts = load_cluster_concepts_from_csv(sparsity_path)
        redundancy_stats[s] = {}

        # Flatten all concepts across clusters to count redundancy
        all_concepts = []
        for concepts in cluster_concepts.values():
            all_concepts.extend(concepts)
        concept_counts = Counter(all_concepts)  # counts across all clusters

        for cluster_name, concepts in cluster_concepts.items():
            unique_func = set(c for c in concepts if label_concept(c) == 'c')
            unique_nonfunc = set(c for c in concepts if label_concept(c) != 'c')

            func_avg_redundancy = (sum(concept_counts[c] for c in unique_func) / len(unique_func)) if unique_func else 0
            nonfunc_avg_redundancy = (sum(concept_counts[c] for c in unique_nonfunc) / len(unique_nonfunc)) if unique_nonfunc else 0

            redundancy_stats[s][cluster_name] = {
                #'func_total': len(unique_func),
                'func_avg_redundancy': func_avg_redundancy,
                #'nonfunc_total': len(unique_nonfunc),
                'nonfunc_avg_redundancy': nonfunc_avg_redundancy
            }

    return redundancy_stats


redundancy_stats=compute_func_nonfunc_redundancy(base_dir)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_func_nonfunc_avg_redundancy(redundancy_stats):
    """
    Plot bar plots of average redundancy for functional vs non-functional concepts.
    
    Parameters:
    - redundancy_stats[sparsity][cluster] -> dict with 'func_avg_redundancy' and 'nonfunc_avg_redundancy'
    """
    sparsities = sorted(redundancy_stats.keys())[:4]  # first 4 sparsities
    clusters = ['c1', 'c2', 'c3']
    n_clusters = len(clusters)
    width = 0.35

    fig, axes = plt.subplots(1, 4, figsize=(20,5), sharey=True)
    axes = np.expand_dims(axes, axis=0)  # make axes[0][i] style

    for i, s in enumerate(sparsities):
        ax = axes[0][i]

        func_avg = [redundancy_stats[s][c]['func_avg_redundancy'] for c in clusters]
        nonfunc_avg = [redundancy_stats[s][c]['nonfunc_avg_redundancy'] for c in clusters]

        x = np.arange(n_clusters)
        bars_func = ax.bar(x - width/2, func_avg, width, color='skyblue', label='Functional')
        bars_nonfunc = ax.bar(x + width/2, nonfunc_avg, width, color='orange', label='Non-Functional')

        # Annotate numbers on top of bars
        for bar in bars_func + bars_nonfunc:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, height + 0.05, f'{height:.2f}',
                    ha='center', va='bottom', fontsize=10)

        ax.set_xticks(x)
        ax.set_xticklabels(clusters)
        ax.set_xlabel("Cluster")
        ax.set_title(f"Sparsity {s}")
        ax.grid(axis='y', linestyle='--', alpha=0.5)

    axes[0][0].set_ylabel("Average Redundancy")
    fig.suptitle("Average Redundancy of Functional vs Non-Functional Concepts", fontsize=16)
    fig.legend(['Functional', 'Non-Functional'], loc='upper right')
    plt.tight_layout(rect=[0,0,1,0.95])
    plt.show()
plot_func_nonfunc_avg_redundancy(redundancy_stats)

In [ ]:
concepts_by_cluster = collect_concepts_per_neuron_by_cluster(base_dir='/workspace/CCE_NLI/BERT/exp/')
def get_pis(root):
    pis =[]
    i=0
    for fldername in sorted(os.listdir(root)):
        print(fldername)
        try:
            pis.append(float(fldername.split("%")[0]))
            i+=1
        except:
            continue
    return pis

pis = get_pis(root='/workspace/CCE_NLI/BERT/exp/CoFi/Run0.25_new/Expls')
def neuron_count_table(concepts_by_cluster):
    """
    Input:
      concepts_by_cluster[method][prune_pct][cluster_id][neuron] = set(concepts)

    Output:
      pandas DataFrame with rows = sparsity levels,
      columns = methods (cofi, lottery_ticket, wanda),
      values = "c1:10 c2:5 c3:1"
    """
    rows = defaultdict(dict)
    
    finallay_neurons =[1024,1022,1022,1022,1020]
    totals={pi:fn for pi,fn in zip(pis, finallay_neurons)}
    #totals={0.0: 1024,0.25:768, 0.4404:573, 0.579: 431, 0.6884:319, 0.7695:256}
    print(totals)
    for method, method_data in concepts_by_cluster.items():
        if method in ['lottery_ticket', 'wanda']:
            total=1024
        for prune_pct, prune_data in method_data.items():
            cluster_counts = []


            if method not in ['lottery_ticket', 'wanda']:
                total = totals[prune_pct]

            for cluster_id, neuron_map in prune_data.items():
                # extract cluster number (Cluster1, Cluster2, ...)
                m = re.search(r"Cluster(\d+)", cluster_id)
                cluster_name = f"c{m.group(1)}" if m else cluster_id

                n_neurons = len(neuron_map)
                cluster_counts.append(f"{cluster_name}:{100*n_neurons/total}")

            # sort c1, c2, c3
            cluster_counts = sorted(
                cluster_counts,
                key=lambda x: int(re.search(r"\d+", x).group())
            )
            print(f"prune_pct: {prune_pct} \t cluster counts: {cluster_counts}")
            rows[prune_pct][method] = " ".join(cluster_counts)

    df = pd.DataFrame.from_dict(rows, orient="index")
    df.index.name = "sparsity"
    df = df.sort_index()

    return df
neuron_count_table(concepts_by_cluster)

In [ ]:
#bowman becomes signficantly more explainable (2% to 23% neurons explainable vs 1.6% to 4.2% or 3.2%) with cofi pruning over lth, wanda

## % and nunmber of functional concepts per cluster, globally, and unique to cluster at each sparsity

In [ ]:
def collect_cluster_concepts_at_sparsity(
    concepts_by_cluster,
    method='lottery_ticket',
    sparsity=0.25,
    cluster_num=3
):
    """
    Returns:
        set of all unique concepts in Cluster{cluster_num}
        for a given sparsity across all neurons
    """
    cluster_key = f"Cluster{cluster_num}"
    all_concepts = set()

    # Get only the requested sparsity
    clusters_at_sparsity = concepts_by_cluster.get(method, {}).get(sparsity, {})

    for cluster_id, neuron_map in clusters_at_sparsity.items():
        if not cluster_id.startswith(cluster_key):
            continue

        for neuron, concepts in neuron_map.items():
            all_concepts.update(concepts)

    return all_concepts
c1_percentfunctional =[]
c2_percentfunctional =[]
c3_percentfunctional =[]
global_func=[]
lenall=[]
funcs=[]
nfuncs=[]
for sp in pis:
    try:
        c3 = collect_cluster_concepts_at_sparsity(concepts_by_cluster, sparsity=sp, method='CoFi', cluster_num=3)
        c2 = collect_cluster_concepts_at_sparsity(concepts_by_cluster, sparsity=sp,method='CoFi', cluster_num=2)
        c1 = collect_cluster_concepts_at_sparsity(concepts_by_cluster, sparsity=sp, method='CoFi', cluster_num=1)
        all_cps_fl = set(c1).union(set(c2)).union(set(c3))
        c,nc, _ = annotate_concepts(all_cps_fl)
        funcs.append(c)
        nfuncs.append(nc)
        onlyc1=set(c1)-set(c2)-set(c3)
        onlyc2=set(c2)-set(c1)-set(c3)
        onlyc3=set(c3)-set(c1)-set(c2)
        c1n,nc1, _ = annotate_concepts(c1)
        c1_percentfunctional.append(100*c1n/(c1n+nc1))
        c2n,nc2, _ = annotate_concepts(c2)
        c2_percentfunctional.append(100*c2n/(c2n+nc2))

        c3n,nc3, _ = annotate_concepts(c3)
        c3_percentfunctional.append(100*c3n/(c3n+nc3))
        lenall.append(len(all_cps_fl))
        global_func.append(100*c/(c+nc))
        print(f'Sparsity {sp}, Percent of foundational concepts in c1 : {100*c1n/(c1n+nc1)}% calculated on {len(set(onlyc1))} concepts')    
        print(f'Sparsity {sp}, Percent of foundational concepts in c2: {100*c2n/(c2n+nc2)}% calculated on {len(set(onlyc2))} concepts')    
        print(f'Sparsity {sp}, Percent of foundational concepts in c3: {100*c3n/(c3n+nc3)}% calculated on {len(set(onlyc3))} concepts')    
        print(f'Sparsity {sp}, Percent of foundational concepts globally : {100*c/(c+nc)}% calculated on {len(set(all_cps_fl))} concepts')
    except:
        continue
        
        
import matplotlib.pyplot as plt

# line plot plotting % of functional concepts
plt.figure(figsize=(6,4))

# Line + points
plt.plot(pis[:4], global_func[:4], marker='o', linewidth=2, label='Functional %')

# Annotate values on points
for x, y in zip(pis[:4], global_func[:4]):
    plt.text(x, y + 0.3, f"{y:.2f}%", ha='center', fontsize=10)

# Labels and title
plt.xlabel("Sparsity")
plt.ylabel("Percent of Functional Concepts")
plt.title("% of Functional Concepts Globally")

# Grid + legend
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()

plt.tight_layout()
plt.show()

# line plot plotting number functional concepts
plt.figure(figsize=(6,4))

# Line + points
plt.plot(pis[:4], lenall[:4], marker='o', linewidth=2, label='Functional %')

# Annotate values on points
for x, y in zip(pis[:4], lenall[:4]):
    plt.text(x, y + 0.3, f"{y:.2f}", ha='center', fontsize=10)

# Labels and title
plt.xlabel("Sparsity")
plt.ylabel("Number of Functional Concepts")
plt.title("Number of unique concepts Globally")

# Grid + legend
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()

plt.tight_layout()
plt.show()


#bar plot counting number of func and nonfunc concepts per sparisty (across all clusters ie globaly)
import matplotlib.pyplot as plt
import numpy as np

# Data
x = np.arange(len(pis[:4]))
width = 0.35

plt.figure(figsize=(7,4))

bars_func = plt.bar(x - width/2, funcs[:4], width, label='Functional')
bars_nonfunc = plt.bar(x + width/2, nfuncs[:4], width, label='Non-Functional')

# Annotate values on bars
for bars in [bars_func, bars_nonfunc]:
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2,
                 height,
                 f"{height:.2f}",
                 ha='center',
                 va='bottom',
                 fontsize=10)

# Axes & labels
plt.xticks(x, pis[:4])
plt.xlabel("Sparsity")
plt.ylabel("Value")
plt.title("Functional vs Non-Functional Concepts")

# Grid + legend
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
concept_counts_by_sparsity(concepts_by_cluster, method="wanda", cluster_num=1)

In [ ]:
concept_counts_by_sparsity(concepts_by_cluster, method="wanda", cluster_num=2)

In [ ]:
concept_counts_by_sparsity(concepts_by_cluster, method="wanda", cluster_num=3)

In [ ]:
concept_counts_by_sparsity(concepts_by_cluster, method="wanda", cluster_num=3, global_=True)

In [ ]:
critical_tokens =  [
    "a", "an", "the", "this", "that", "these", "those",
    "some", "any", "each", "every", "no",
    "i", "you", "he", "she", "it", "they",
    "me", "him", "her", "them",
    "his", "hers", "their", "its",
    "someone", "something", "nobody",
    "in", "on", "at", "by", "for", "from", "to", "with",
    "about", "into", "through", "over", "under",
    "between", "near", "inside", "outside",
    "and", "or", "but", "because", "while", "if", "though", "although", "so",
    "is", "am", "are", "was", "were", "be", "being", "been",
    "do", "does", "did",
    "have", "has", "had",
    "can", "could", "will", "would", "should", "may", "might", "must",
    "not", "none", "never", "nothing", "least", "most", "all",
    "there", "here", "just", "only", "also", "very",
    ".", ",", ":", ";", "!", "?"
]


def label_concept(concept: str):
    """
    Returns 'c' for critical or 'nc' for non-critical
    """
    # POS tags are always critical
    if ":tag:" in concept:
        return "c"

    # Overlap / structural features
    if concept.startswith("oth:"):
        return "c"

    # Token-level concepts
    if ":tok:" in concept:
        token = concept.split(":tok:")[-1]
        return "c" if token in critical_tokens else "nc"

    # Default fallback
    return "nc"


def annotate_concepts(concept_set):
    annotated = {}
    num_critical = 0
    num_noncritical = 0

    for concept in sorted(concept_set):
        label = label_concept(concept)
        annotated[concept] = label

        if label == "c":
            num_critical += 1
        else:
            num_noncritical += 1


    return num_critical, num_noncritical, annotated

annotate_concepts(all_cps_fl)

In [ ]:
def collect_cluster_concepts_at_sparsity(
    concepts_by_cluster,
    method='lottery_ticket',
    sparsity=0.25,
    cluster_num=3
):
    """
    Returns:
        set of all unique concepts in Cluster{cluster_num}
        for a given sparsity across all neurons
    """
    cluster_key = f"Cluster{cluster_num}"
    all_concepts = set()

    # Get only the requested sparsity
    clusters_at_sparsity = concepts_by_cluster.get(method, {}).get(sparsity, {})

    for cluster_id, neuron_map in clusters_at_sparsity.items():
        if not cluster_id.startswith(cluster_key):
            continue

        for neuron, concepts in neuron_map.items():
            all_concepts.update(concepts)

    return all_concepts
c1_percentfunctional =[]
c2_percentfunctional =[]
c3_percentfunctional =[]
global_func=[]
for sp in pis:
    c3 = collect_cluster_concepts_at_sparsity(concepts_by_cluster, sparsity=sp, method='CoFi', cluster_num=3)
    c2 = collect_cluster_concepts_at_sparsity(concepts_by_cluster, sparsity=sp,method='CoFi', cluster_num=2)
    c1 = collect_cluster_concepts_at_sparsity(concepts_by_cluster, sparsity=sp, method='CoFi', cluster_num=1)
    all_cps_fl = set(c1).union(set(c2)).union(set(c3))
    c,nc, _ = annotate_concepts(all_cps_fl)
    onlyc1=set(c1)-set(c2)-set(c3)
    onlyc2=set(c2)-set(c1)-set(c3)
    onlyc3=set(c3)-set(c1)-set(c2)
    c1n,nc1, _ = annotate_concepts(c1)
    c1_percentfunctional.append(100*c1n/(c1n+nc1))
    c2n,nc2, _ = annotate_concepts(c2)
    c2_percentfunctional.append(100*c2n/(c2n+nc2))
    
    c3n,nc3, _ = annotate_concepts(c3)
    c3_percentfunctional.append(100*c3n/(c3n+nc3))
    
    global_func.append(100*c/(c+nc))
    print(f'Sparsity {sp}, Percent of foundational concepts in c1 : {100*c1n/(c1n+nc1)}% calculated on {len(set(c1))} concepts')    
    print(f'Sparsity {sp}, Percent of foundational concepts in c2: {100*c2n/(c2n+nc2)}% calculated on {len(set(c2))} concepts')    
    print(f'Sparsity {sp}, Percent of foundational concepts in c3: {100*c3n/(c3n+nc3)}% calculated on {len(set(c3))} concepts')    
    print(f'Sparsity {sp}, Percent of foundational concepts globally : {100*c/(c+nc)}% calculated on {len(set(all_cps_fl))} concepts')
    


In [ ]:
c1_percentfunctional
c2_percentfunctional
c3_percentfunctional
c3_percentfunctional,c2_percentfunctional,c1_percentfunctional
global_func

In [ ]:
all_cps_fl

In [ ]:
onlyc2

In [ ]:
onlyc1 #cps at c1 but not c2,c3 (at all sparsities)